In [1]:
from pathlib import Path
import random
import urllib.parse as up

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
import pandas as pd
from tqdm import tqdm

BASE_DIR = Path("..").resolve()
PROCESSED_DIR = BASE_DIR / "data" / "processed"

device = "cuda" if torch.cuda.is_available() else "cpu"
device


c:\Users\Moritz\miniconda3\envs\phishing-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'cuda'

In [2]:
# Testdaten laden
test_df = pd.read_csv(PROCESSED_DIR / "urls_test.csv")
print(test_df["label"].value_counts())

# nur Phishing-URLs (label = 1)
phish_df = test_df[test_df["label"] == 1].reset_index(drop=True)
print("Phishing-URLs im Test:", len(phish_df))

# kleine Stichprobe für schnelle Experimente
N = 2000
if len(phish_df) > N:
    phish_df = phish_df.sample(n=N, random_state=42).reset_index(drop=True)

len(phish_df), phish_df.head()


label
0    78585
1    31585
Name: count, dtype: int64
Phishing-URLs im Test: 31585


(2000,
                                             url  label  source
 0                myglyko.com/yyahook/yahoo.html      1  github
 1                              helsby.biz/wtcui      1  github
 2               lalarabbit.web.fc2.com/j8fn3rg3      1  github
 3  jamalacademy.com/wp-content/dropbox=id45342/      1  github
 4                      spicythaicafe.com/4I9bwO      1  github)

In [3]:
tokenizer = DistilBertTokenizerFast.from_pretrained(BASE_DIR / "models" / "bert")
model = DistilBertForSequenceClassification.from_pretrained(BASE_DIR / "models" / "bert").to(device)
model.eval()


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [4]:
class URLDataset(Dataset):
    def __init__(self, urls, tokenizer, max_len=64):
        self.urls = [str(u) for u in urls]
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.urls)

    def __getitem__(self, idx):
        url = self.urls[idx]
        enc = self.tokenizer(
            url,
            add_special_tokens=True,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
        }


def bert_predict_urls(urls, tokenizer, model, device, batch_size=64):
    ds = URLDataset(urls, tokenizer)
    dl = DataLoader(ds, batch_size=batch_size)

    preds, probs = [], []

    with torch.no_grad():
        for batch in tqdm(dl, total=len(dl), desc="BERT eval"):
            input_ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)

            out = model(input_ids, attention_mask=mask)
            logits = out.logits
            prob = torch.softmax(logits, dim=1)[:, 1]  # Klasse 1 = phishing

            preds.extend(torch.argmax(logits, dim=1).cpu().tolist())
            probs.extend(prob.cpu().tolist())

    return preds, probs


In [5]:
def ensure_http(url: str) -> str:
    if url.startswith(("http://", "https://")):
        return url
    return "http://" + url

def add_noise_subdomain(url: str) -> str:
    u = ensure_http(url)
    parsed = up.urlsplit(u)
    host = parsed.netloc
    parts = host.split(".")
    # subdomain einfügen
    parts = ["secure-check"] + parts
    new_host = ".".join(parts)
    return up.urlunsplit((parsed.scheme, new_host, parsed.path, parsed.query, parsed.fragment))

def add_tracking_params(url: str) -> str:
    u = ensure_http(url)
    parsed = up.urlsplit(u)
    q = parsed.query
    extra = "utm_source=email&utm_campaign=security_update"
    new_q = extra if not q else q + "&" + extra
    return up.urlunsplit((parsed.scheme, parsed.netloc, parsed.path, new_q, parsed.fragment))

def encode_path(url: str) -> str:
    u = ensure_http(url)
    parsed = up.urlsplit(u)
    encoded_path = up.quote(parsed.path, safe="/")
    return up.urlunsplit((parsed.scheme, parsed.netloc, encoded_path, parsed.query, parsed.fragment))

def homoglyph_paypal(url: str) -> str:
    # paypal -> paypaI (großes i) / login -> Iogin
    u = str(url)
    u2 = u.replace("paypal", "paypaI").replace("login", "Iogin")
    return u2

def apply_attack(url: str, attack_type: str) -> str:
    if attack_type == "subdomain":
        return add_noise_subdomain(url)
    if attack_type == "tracking":
        return add_tracking_params(url)
    if attack_type == "encode":
        return encode_path(url)
    if attack_type == "homoglyph":
        return homoglyph_paypal(url)
    return url  # fallback


In [6]:
attack_types = ["subdomain", "tracking", "encode", "homoglyph"]

rows = []
for idx, row in phish_df.iterrows():
    orig = row["url"]
    label = row["label"]  # sollte 1 sein
    for atk in attack_types:
        adv = apply_attack(orig, atk)
        rows.append({
            "url_orig": orig,
            "url_adv": adv,
            "label": label,
            "attack": atk,
        })

adv_df = pd.DataFrame(rows)
len(adv_df), adv_df.head()


(8000,
                          url_orig  \
 0  myglyko.com/yyahook/yahoo.html   
 1  myglyko.com/yyahook/yahoo.html   
 2  myglyko.com/yyahook/yahoo.html   
 3  myglyko.com/yyahook/yahoo.html   
 4                helsby.biz/wtcui   
 
                                              url_adv  label     attack  
 0  http://secure-check.myglyko.com/yyahook/yahoo....      1  subdomain  
 1  http://myglyko.com/yyahook/yahoo.html?utm_sour...      1   tracking  
 2              http://myglyko.com/yyahook/yahoo.html      1     encode  
 3                     myglyko.com/yyahook/yahoo.html      1  homoglyph  
 4               http://secure-check.helsby.biz/wtcui      1  subdomain  )

In [7]:
# Vorhersage für Original-URLs (Phishing)
orig_preds, orig_probs = bert_predict_urls(phish_df["url"].tolist(), tokenizer, model, device)
phish_df["bert_pred"] = orig_preds
phish_df["bert_proba"] = orig_probs

# nur korrekt erkannte Phishing-URLs (TP), auf denen wir Attacken testen wollen
phish_tp = phish_df[phish_df["bert_pred"] == 1].reset_index(drop=True)
print("Anzahl korrekter Phishing-Detektionen (BERT):", len(phish_tp))


BERT eval: 100%|██████████| 32/32 [00:01<00:00, 16.59it/s]

Anzahl korrekter Phishing-Detektionen (BERT): 1901


In [8]:
rows = []
for idx, row in phish_tp.iterrows():
    orig = row["url"]
    for atk in attack_types:
        adv = apply_attack(orig, atk)
        rows.append({
            "url_orig": orig,
            "url_adv": adv,
            "label": 1,
            "attack": atk,
        })

adv_df = pd.DataFrame(rows)
len(adv_df), adv_df.head()



(7604,
                          url_orig  \
 0  myglyko.com/yyahook/yahoo.html   
 1  myglyko.com/yyahook/yahoo.html   
 2  myglyko.com/yyahook/yahoo.html   
 3  myglyko.com/yyahook/yahoo.html   
 4                helsby.biz/wtcui   
 
                                              url_adv  label     attack  
 0  http://secure-check.myglyko.com/yyahook/yahoo....      1  subdomain  
 1  http://myglyko.com/yyahook/yahoo.html?utm_sour...      1   tracking  
 2              http://myglyko.com/yyahook/yahoo.html      1     encode  
 3                     myglyko.com/yyahook/yahoo.html      1  homoglyph  
 4               http://secure-check.helsby.biz/wtcui      1  subdomain  )

In [9]:
adv_preds, adv_probs = bert_predict_urls(adv_df["url_adv"].tolist(), tokenizer, model, device)
adv_df["bert_pred"] = adv_preds
adv_df["bert_proba"] = adv_probs


BERT eval: 100%|██████████| 119/119 [00:06<00:00, 17.96it/s]


In [10]:
# Für jede Attacke getrennt auswerten
results = []

for atk in attack_types:
    sub = adv_df[adv_df["attack"] == atk]
    n_total = len(sub)
    n_missed = (sub["bert_pred"] == 0).sum()  # jetzt FN
    success_rate = n_missed / n_total if n_total > 0 else 0.0
    results.append({
        "attack": atk,
        "n_total": n_total,
        "n_missed": n_missed,
        "adv_success_rate": success_rate,
    })

pd.DataFrame(results)


,attack,n_total,n_missed,adv_success_rate
0,subdomain,1901,1,0.000526
1,tracking,1901,21,0.011047
2,encode,1901,69,0.036297
3,homoglyph,1901,7,0.003682


In [11]:
# Beispiel: 5 erfolgreiche Angriffe pro Attack-Typ
examples = []

for atk in attack_types:
    sub = adv_df[(adv_df["attack"] == atk) & (adv_df["bert_pred"] == 0)]
    ex = sub.head(5)[["attack", "url_orig", "url_adv", "bert_proba"]]
    examples.append(ex)

pd.concat(examples, ignore_index=True)


,attack,url_orig,url_adv,bert_proba
0,subdomain,jiang.aspx?e1=,http://secure-check.jiang.aspx?e1=,0.195993
1,tracking,helsby.biz/wtcui,http://helsby.biz/wtcui?utm_source=email&utm_c...,0.497136
2,tracking,deploy.html,http://deploy.html?utm_source=email&utm_campai...,0.097230
3,tracking,6b8a953b2bf7788063d5-6e453f33ecbb90f11a62a5c37...,http://6b8a953b2bf7788063d5-6e453f33ecbb90f11a...,0.159267
4,tracking,eternitymobiles.com,http://eternitymobiles.com?utm_source=email&ut...,0.093133
5,tracking,camorg.net/works/docs/basement,http://camorg.net/works/docs/basement?utm_sour...,0.290834
6,encode,helsby.biz/wtcui,http://helsby.biz/wtcui,0.077320
7,encode,deploy.html,http://deploy.html,0.110958
8,encode,darkhollowcoffee.com/xlbps,http://darkhollowcoffee.com/xlbps,0.044197
9,encode,www.speedagecommercials.net/online.php,http://www.speedagecommercials.net/online.php,0.378386
